In [1]:
import pandas as pd

In [5]:
movies=pd.read_csv("../../data/raw/movies_raw_enriched.csv")

In [6]:
movies.head(1)

,movie_id,title,overview,release_date,genres,keywords,runtime,vote_average,vote_count,poster_url,cast,director
0,11,Star Wars,Princess Leia is captured and held hostage by ...,1977-05-25,"[""Adventure"", ""Action"", ""Science Fiction""]","[""empire"", ""galaxy"", ""rebellion"", ""android"", ""...",121,8.208,23043,https://image.tmdb.org/t/p/w500fai0rspsNeJCS69...,Mark Hamill | Harrison Ford | Carrie Fisher | ...,George Lucas


In [ ]:
movies.shape

(35001, 12)

In [8]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 35001 entries, 0 to 35000
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   movie_id      35001 non-null  int64  
 1   title         35001 non-null  str    
 2   overview      31659 non-null  str    
 3   release_date  34959 non-null  str    
 4   genres        35001 non-null  str    
 5   keywords      35001 non-null  str    
 6   runtime       35001 non-null  int64  
 7   vote_average  35001 non-null  float64
 8   vote_count    35001 non-null  int64  
 9   poster_url    31452 non-null  str    
 10  cast          33343 non-null  str    
 11  director      33994 non-null  str    
dtypes: float64(1), int64(3), str(8)
memory usage: 18.8 MB


In [9]:
# Actual null values
null_counts = movies.isna().sum()

# Empty or whitespace-only strings
empty_counts = pd.Series({
    col: (
        movies[col]
        .astype("string")
        .str.strip()
        .eq("")
        .fillna(False)
        .sum()
    )
    if movies[col].dtype in ("object", "str", "string")
    else 0
    for col in movies.columns
})

missing_summary = pd.DataFrame({
    "Null Count": null_counts,
    "Empty String Count": empty_counts,
})

missing_summary["Total Missing"] = (
    missing_summary["Null Count"]
    + missing_summary["Empty String Count"]
)

missing_summary["Missing %"] = (
    missing_summary["Total Missing"] / len(movies) * 100
).round(2)

missing_summary.sort_values(
    "Total Missing",
    ascending=False
)

,Null Count,Empty String Count,Total Missing,Missing %
poster_url,3549,0,3549,10.14
overview,3342,0,3342,9.55
cast,1658,0,1658,4.74
director,1007,0,1007,2.88
release_date,42,0,42,0.12
title,0,0,0,0.00
movie_id,0,0,0,0.00
genres,0,0,0,0.00
vote_average,0,0,0,0.00
runtime,0,0,0,0.00


In [10]:
# Check exact duplicate rows
exact_duplicates = movies.duplicated().sum()

# Check duplicate movie IDs
duplicate_ids = movies["movie_id"].duplicated().sum()

print(f"Exact duplicate rows: {exact_duplicates:,}")
print(f"Duplicate movie IDs: {duplicate_ids:,}")

Exact duplicate rows: 0
Duplicate movie IDs: 0


In [13]:
model_columns = [
    "movie_id",
    "title",
    "overview",
    "genres",
    "keywords",
    "cast",
    "director",
]

model_data = movies[model_columns].copy()

print("Original dataset shape:", movies.shape)
print("Model working dataset shape:", model_data.shape)

model_data.head()

Original dataset shape: (35001, 12)
Model working dataset shape: (35001, 7)


,movie_id,title,overview,genres,keywords,cast,director
0,11,Star Wars,Princess Leia is captured and held hostage by ...,"[""Adventure"", ""Action"", ""Science Fiction""]","[""empire"", ""galaxy"", ""rebellion"", ""android"", ""...",Mark Hamill | Harrison Ford | Carrie Fisher | ...,George Lucas
1,12,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...","[""Animation"", ""Family"", ""Adventure""]","[""fish"", ""sydney, australia"", ""parent child re...",Albert Brooks | Ellen DeGeneres | Alexander Go...,Andrew Stanton
2,13,Forrest Gump,A man with a low IQ has accomplished great thi...,"[""Comedy"", ""Drama"", ""Romance""]","[""new year's eve"", ""vietnam war"", ""vietnam vet...",Tom Hanks | Robin Wright | Gary Sinise | Sally...,Robert Zemeckis
3,14,American Beauty,"Lester Burnham, a depressed suburban father in...","[""Drama""]","[""estate agent"", ""adultery"", ""coming out"", ""fi...",Kevin Spacey | Annette Bening | Thora Birch | ...,Sam Mendes
4,15,Citizen Kane,Newspaper magnate Charles Foster Kane is taken...,"[""Mystery"", ""Drama""]","[""media tycoon"", ""florida"", ""art collector"", ""...",Orson Welles | Joseph Cotten | Dorothy Comingo...,Orson Welles


In [15]:
text_columns = [
    "overview",
    "genres",
    "keywords",
    "cast",
    "director",
]

# Treat both NaN and empty/whitespace-only strings as missing
missing_text = (
    model_data[text_columns]
    .isna()
    | model_data[text_columns].apply(
        lambda col: col.astype("string").str.strip().eq("")
    )
)

print("Total movies:", len(model_data))

print("\nMovies missing each text field:")
print(missing_text.sum().sort_values(ascending=False))

complete_mask = ~missing_text.any(axis=1)

print("\nMovies with all five text fields:", complete_mask.sum())
print("Movies that would be removed:", (~complete_mask).sum())
print(
    "Percentage retained:",
    round(complete_mask.mean() * 100, 2),
    "%"
)

Total movies: 35001

Movies missing each text field:
overview    3342
cast        1658
director    1007
genres         0
keywords       0
dtype: uint64[pyarrow]

Movies with all five text fields: 29820
Movies that would be removed: 5181
Percentage retained: 85.2 %


In [16]:
text_columns = [
    "overview",
    "genres",
    "keywords",
    "cast",
    "director",
]

model_data = model_data.dropna(subset=text_columns).copy()

print("Original dataset:", movies.shape)
print("Modeling dataset:", model_data.shape)
print("Remaining missing values:", model_data[text_columns].isna().sum().sum())

Original dataset: (35001, 12)
Modeling dataset: (29820, 7)
Remaining missing values: 0


In [20]:
model_data[
    ["title", "overview", "genres", "keywords", "cast", "director"]
].head(5)

,title,overview,genres,keywords,cast,director
0,Star Wars,Princess Leia is captured and held hostage by ...,"[""Adventure"", ""Action"", ""Science Fiction""]","[""empire"", ""galaxy"", ""rebellion"", ""android"", ""...",Mark Hamill | Harrison Ford | Carrie Fisher | ...,George Lucas
1,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...","[""Animation"", ""Family"", ""Adventure""]","[""fish"", ""sydney, australia"", ""parent child re...",Albert Brooks | Ellen DeGeneres | Alexander Go...,Andrew Stanton
2,Forrest Gump,A man with a low IQ has accomplished great thi...,"[""Comedy"", ""Drama"", ""Romance""]","[""new year's eve"", ""vietnam war"", ""vietnam vet...",Tom Hanks | Robin Wright | Gary Sinise | Sally...,Robert Zemeckis
3,American Beauty,"Lester Burnham, a depressed suburban father in...","[""Drama""]","[""estate agent"", ""adultery"", ""coming out"", ""fi...",Kevin Spacey | Annette Bening | Thora Birch | ...,Sam Mendes
4,Citizen Kane,Newspaper magnate Charles Foster Kane is taken...,"[""Mystery"", ""Drama""]","[""media tycoon"", ""florida"", ""art collector"", ""...",Orson Welles | Joseph Cotten | Dorothy Comingo...,Orson Welles


In [21]:
sample = model_data.iloc[0]

for column in ["genres", "keywords", "cast", "director"]:
    print(f"\n{column.upper()}:")
    print(sample[column])


GENRES:
["Adventure", "Action", "Science Fiction"]

KEYWORDS:
["empire", "galaxy", "rebellion", "android", "hermit", "smuggling (contraband)", "freedom", "rebel", "rescue mission", "space", "planet", "desert", "super power", "oppression", "space opera", "wizard", "totalitarianism", "quest", "powerful", "tuwaderalit"]

CAST:
Mark Hamill | Harrison Ford | Carrie Fisher | Peter Cushing | Alec Guinness

DIRECTOR:
George Lucas


In [ ]:
import json

def parse_list(value):
    try:
        items = json.loads(value)
        if isinstance(items, list):
            return " ".join(str(item) for item in items)
    except (json.JSONDecodeError, TypeError):
        pass

    return str(value)


model_data["genres_text"] = model_data["genres"].apply(parse_list)
model_data["keywords_text"] = model_data["keywords"].apply(parse_list)

model_data["tags"] = (
    model_data["overview"].astype(str) + " " +
    model_data["genres_text"] + " " +
    model_data["keywords_text"] + " " +
    model_data["cast"].astype(str).str.replace("|", " ", regex=False) + " " +
    model_data["director"].astype(str)
).str.lower().str.strip()

model_data[["title", "tags"]].head(3)
model_data=model_data[['movie_id','title','tags']]
model_data.head(1)

,title,tags
0,Star Wars,princess leia is captured and held hostage by ...
1,Finding Nemo,"nemo, an adventurous young clownfish, is unexp..."
2,Forrest Gump,a man with a low iq has accomplished great thi...


In [28]:
print("Dataset shape:", model_data.shape)
print("Missing tags:", model_data["tags"].isna().sum())
print("Empty tags:", model_data["tags"].str.strip().eq("").sum())

model_data[["title", "tags"]].sample(5, random_state=42)

Dataset shape: (29820, 3)
Missing tags: 0
Empty tags: 0


,title,tags
5497,House of Traps,"it all started with the five venoms, the inter..."
579,The Hunt for Red October,a new technologically-superior soviet nuclear ...
279,Grease,australian good girl sandy and greaser danny f...
16197,Y's Experience,x's unconditional love for fascinating writer ...
25729,Death Whisperer,when a remote village is plagued by a deadly c...


In [91]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

In [92]:
tfidf_matrix = tfidf.fit_transform(model_data["tags"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))
print("Matrix type:", type(tfidf_matrix))

TF-IDF matrix shape: (29820, 5000)
Vocabulary size: 5000
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>


In [93]:
from sklearn.metrics.pairwise import cosine_similarity
movie_index = 0

similarity_scores = cosine_similarity(
    tfidf_matrix[movie_index],
    tfidf_matrix
).flatten()

print("Number of similarity scores:", len(similarity_scores))
print("Similarity with itself:", similarity_scores[movie_index])

Number of similarity scores: 29820
Similarity with itself: 1.0000000000000002


In [94]:
def recommend_movies(movie_id, n=5):
    matches = model_data.index[
        model_data["movie_id"] == movie_id
    ]

    if len(matches) == 0:
        return f"Movie ID {movie_id} not found."

    movie_index = matches[0]

    similarity_scores = cosine_similarity(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).flatten()

    ranked_indices = np.argsort(similarity_scores)[::-1]
    ranked_indices = [
        idx for idx in ranked_indices
        if idx != movie_index
    ][:n]

    recommendations = model_data.iloc[ranked_indices][
        ["movie_id", "title"]
    ].copy()

    recommendations["similarity"] = [
        similarity_scores[idx] for idx in ranked_indices
    ]

    return recommendations.reset_index(drop=True)

In [96]:
recommend_movies(847742)

,movie_id,title,similarity
0,706872,Drishyam 2,0.481181
1,244049,Drishyam,0.259981
2,280294,Appu,0.256032
3,453093,Dhruvasangamam,0.202190
4,1576369,Bharathanatyam 2 Mohiniyattam,0.190568


In [48]:
model_data[
    model_data["title"].str.contains("Drishyam", case=False, na=False)
][["movie_id", "title", "tags"]]

,movie_id,title,tags
9869,244049,Drishyam,georgekutty lives a happy life with his wife a...
13598,352173,Drishyam,when his family stares down the barrel of crim...
20504,706872,Drishyam 2,seven years after the events of the first film...
22217,847742,Drishyam 3,"to protect his family and their dark secret, g..."
24661,1029827,Drishyam 2,7 years after the case related to vijay salgao...
27881,1269325,Drishyam: The Conclusion,"years after outsmarting the police, vijay salg..."


In [49]:
# Reset the index so DataFrame row positions match TF-IDF rows
model_data = model_data.reset_index(drop=True)

# Rebuild TF-IDF using the same row order
tfidf_matrix = tfidf.fit_transform(model_data["tags"])

print("DataFrame shape:", model_data.shape)
print("TF-IDF matrix shape:", tfidf_matrix.shape)

DataFrame shape: (29820, 3)
TF-IDF matrix shape: (29820, 5000)


In [61]:
# Normalize titles for comparison without changing their displayed names
normalized_titles = model_data["title"].str.strip().str.casefold()

duplicate_title_mask = normalized_titles.duplicated(keep=False)

print("Rows involved in duplicate titles:", duplicate_title_mask.sum())
print("Number of duplicate title groups:",
      normalized_titles[duplicate_title_mask].nunique())

model_data.loc[
    duplicate_title_mask,
    ["movie_id", "title"]
].sort_values("title").head(30)

Rows involved in duplicate titles: 2497
Number of duplicate title groups: 1095


,movie_id,title
4992,55627,100 Days
23223,1135433,100 Days
4745,52011,13 Assassins
5190,58857,13 Assassins
12333,323376,A Love Story
21673,970644,A Love Story
4058,42726,A Man and a Woman
13642,384127,A Man and a Woman
2998,28162,A Matter of Life and Death
25787,1418982,A Matter of Life and Death


In [62]:
title_to_check = "13 Assassins"

movies.loc[
    movies["title"].str.strip().str.casefold()
    == title_to_check.casefold(),
    [
        "movie_id",
        "title",
        "release_date",
        "genres",
        "overview",
        "vote_average",
        "vote_count",
    ],
]

,movie_id,title,release_date,genres,overview,vote_average,vote_count
4760,52011,13 Assassins,1963-12-07,"[""Adventure"", ""Drama"", ""Action""]",A sadistic Daimyo (feudal lord) rapes a woman ...,7.5,45
5208,58857,13 Assassins,2010-09-25,"[""Adventure"", ""Drama"", ""Action""]",A bravado period action film set at the end of...,7.3,1332


In [63]:
# Create a lookup table for movie metadata
movie_metadata = movies[
    ["movie_id", "title", "release_date", "poster_url"]
].copy()

movie_metadata["release_year"] = pd.to_datetime(
    movie_metadata["release_date"], errors="coerce"
).dt.year

movie_metadata = movie_metadata[
    ["movie_id", "release_year", "poster_url"]
]

In [97]:

def recommend_movies(movie_title, n=5):
    matches = model_data[
        model_data["title"].str.strip().str.casefold()
        == movie_title.strip().casefold()
    ]

    if matches.empty:
        return f"No movie found with the title '{movie_title}'."

    if len(matches) > 1:
        return matches[["movie_id", "title"]].merge(
            movie_metadata[["movie_id", "release_year"]],
            on="movie_id",
            how="left"
        ).reset_index(drop=True)

    movie_index = matches.index[0]

    similarity_scores = cosine_similarity(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).flatten()

    ranked_indices = np.argsort(similarity_scores)[::-1]
    ranked_indices = [
        idx for idx in ranked_indices if idx != movie_index
    ][:n]

    recommendations = model_data.iloc[ranked_indices][
        ["movie_id", "title"]
    ].copy()

    recommendations["similarity"] = [
        similarity_scores[idx] for idx in ranked_indices
    ]

    recommendations = recommendations.merge(
        movie_metadata,
        on="movie_id",
        how="left"
    )

    return recommendations[
        ["movie_id", "title", "release_year", "similarity", "poster_url"]
    ].reset_index(drop=True)


In [98]:
recommend_movies("Drishyam")

,movie_id,title,release_year
0,244049,Drishyam,2013.0
1,352173,Drishyam,2015.0


In [99]:

def recommend_movies(movie_title, movie_id=None, n=5):
    matches = model_data[
        model_data["title"].str.strip().str.casefold()
        == movie_title.strip().casefold()
    ]

    if matches.empty:
        return f"No movie found with the title '{movie_title}'."

    if movie_id is not None:
        matches = matches[matches["movie_id"] == movie_id]

        if matches.empty:
            return f"No matching movie ID {movie_id} for '{movie_title}'."

    elif len(matches) > 1:
        return matches[["movie_id", "title"]].merge(
            movie_metadata[["movie_id", "release_year"]],
            on="movie_id",
            how="left"
        ).reset_index(drop=True)

    movie_index = matches.index[0]

    similarity_scores = cosine_similarity(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).flatten()

    ranked_indices = np.argsort(similarity_scores)[::-1]
    ranked_indices = [
        idx for idx in ranked_indices if idx != movie_index
    ][:n]

    recommendations = model_data.iloc[ranked_indices][
        ["movie_id", "title"]
    ].copy()

    recommendations["similarity"] = [
        similarity_scores[idx] for idx in ranked_indices
    ]

    recommendations = recommendations.merge(
        movie_metadata,
        on="movie_id",
        how="left"
    )

    return recommendations[
        ["movie_id", "title", "release_year", "similarity", "poster_url"]
    ].reset_index(drop=True)


In [100]:
recommend_movies("Drishyam", movie_id=244049)

,movie_id,title,release_year,similarity,poster_url
0,352173,Drishyam,2015.0,0.321540,https://image.tmdb.org/t/p/w500gIClWRv5OSe8rl5...
1,706872,Drishyam 2,2021.0,0.264800,https://image.tmdb.org/t/p/w5008RJBCUGE27LX06t...
2,652783,Sheep Without a Shepherd,2019.0,0.261115,https://image.tmdb.org/t/p/w5001pjSsjhkv4tiFe2...
3,847742,Drishyam 3,2026.0,0.259981,https://image.tmdb.org/t/p/w500456FlpiU1iOBnaO...
4,55696,The Elevator,1999.0,0.253288,https://image.tmdb.org/t/p/w500tIVEUIw7gdziDgI...


In [101]:
recommend_movies("The Notebook")

,movie_id,title,release_year,similarity,poster_url
0,640306,Interrupted Song,1960.0,0.331060,https://image.tmdb.org/t/p/w500b4rrec9s6iL73ev...
1,729944,The Stepmother,1971.0,0.225574,https://image.tmdb.org/t/p/w5006eCIAjumcQ8oF7n...
2,19659,Yamato,2005.0,0.221970,https://image.tmdb.org/t/p/w500tTCUAlGakuDkblt...
3,61807,Ethan Frome,1993.0,0.208956,https://image.tmdb.org/t/p/w500wsXL08E4gQHWW5p...
4,1475174,Amruthamathi,2025.0,0.208255,https://image.tmdb.org/t/p/w500yWJJ3erDZJcluMQ...


In [75]:
notebook_tags = model_data.loc[
    model_data["title"].str.strip().str.casefold() == "the notebook",
    "tags"
]

for tags in notebook_tags:
    print(tags)

an epic love story centered around an older man who reads aloud to a woman with alzheimer's. from a faded notebook, the old man's words bring to life the story about a couple who is separated by world war ii, and is then passionately reunited, seven years later, after they have taken different paths. romance drama secret love soulmates based on novel or book love of one's life fight poem river sadness class dementia tears candle mailbox valentine's day romantic ryan gosling   rachel mcadams   gena rowlands   james garner   joan allen nick cassavetes


In [76]:
recommend_movies("The Notebook", n=10)

,movie_id,title,release_year,similarity,poster_url
0,640306,Interrupted Song,1960.0,0.331060,https://image.tmdb.org/t/p/w500b4rrec9s6iL73ev...
1,729944,The Stepmother,1971.0,0.225574,https://image.tmdb.org/t/p/w5006eCIAjumcQ8oF7n...
2,19659,Yamato,2005.0,0.221970,https://image.tmdb.org/t/p/w500tTCUAlGakuDkblt...
3,61807,Ethan Frome,1993.0,0.208956,https://image.tmdb.org/t/p/w500wsXL08E4gQHWW5p...
4,1475174,Amruthamathi,2025.0,0.208255,https://image.tmdb.org/t/p/w500yWJJ3erDZJcluMQ...
5,1156594,Our Fault,2025.0,0.197745,https://image.tmdb.org/t/p/w500yzqHt4m1SeY9FbP...
6,1261555,Promised Hearts,2025.0,0.196766,https://image.tmdb.org/t/p/w500shQW1KFEAEHCX9G...
7,304337,Pallavi Anu Pallavi,1983.0,0.195352,https://image.tmdb.org/t/p/w500w3XHqLxSk4fys80...
8,410429,Verbal Messages,1991.0,0.191852,https://image.tmdb.org/t/p/w500iT0oei2khnwUdNU...
9,93523,Beder Meye Josna,1989.0,0.189255,https://image.tmdb.org/t/p/w500cG5RywylnNyLUnS...


In [102]:
recommend_movies("Saiyaara")

,movie_id,title,release_year,similarity,poster_url
0,1721615,Sehre Ke Phool,1973.0,0.246552,NaN
1,935141,Zoebaida,1940.0,0.246055,https://image.tmdb.org/t/p/w500AnIPWT1FUGvMmNv...
2,271758,Özleyiş,1961.0,0.231209,https://image.tmdb.org/t/p/w500Ab7aabgBS2GTnCH...
3,1212121,Latin Blood – The Ballad of Ney Matogrosso,2025.0,0.230927,https://image.tmdb.org/t/p/w500vWhWbA4w4Pn8WXo...
4,617304,Gangster vs State,2019.0,0.226428,https://image.tmdb.org/t/p/w500wPiC9vIr3G17iFZ...


In [80]:
model_data[
    model_data["title"].str.contains(
        "Saiyaara", case=False, na=False
    )
][["movie_id", "tags"]]

,movie_id,tags
24360,1241634,two artistic souls find harmony through music ...
